## Simulations with heavy-tailed distributions

In [ ]:
import numpy as np
import time
import torch
import pandas as pd
from sklearn.manifold import TSNE, Isomap, trustworthiness
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances
import umap.umap_ as umap
from Gini_MDS import GiniMDS



# Simulated data

def generate_heavy_tailed_data(n=500):
    d = 6
    X = np.zeros((n, d))
    for j in range(d):
        if j == 0:
            x = np.random.normal(0, 1, n)
            outliers = np.random.choice(n, int(0.05*n), replace=False)
            x[outliers] += np.random.normal(0, 10, len(outliers))
        elif j == 1:
            x = np.random.standard_cauchy(n)
        elif j == 2:
            x = np.random.weibull(a=0.5, size=n)
        elif j == 3:
            x = np.random.pareto(a=2.0, size=n)
        elif j == 4:
            x = np.random.standard_t(df=2, size=n)
        else:
            x = np.random.lognormal(mean=0, sigma=1.5, size=n)
        X[:, j] = x

    # Robust standardization (median)
    X = (X - np.median(X, axis=0)) / (np.std(X, axis=0) + 1e-8)
    y = (X[:, 0] > np.median(X[:, 0])).astype(int)
    return X, y



# Metrics: trust, silhouette & neighbors

def compute_metrics(X, Z, y):
    trust = trustworthiness(X, Z, n_neighbors=5)
    sil = silhouette_score(Z, y)
    nbrs = NearestNeighbors(n_neighbors=10).fit(Z)
    indices = nbrs.kneighbors(return_distance=False)
    nn_acc = np.mean([
        np.mean(y[neighbors] == y[i])
        for i, neighbors in enumerate(indices)
    ])
    return trust, sil, nn_acc



# Distances (torch tensors)

def pairwise_distances_torch(X):
    diff = X.unsqueeze(1) - X.unsqueeze(0)
    return torch.sqrt(torch.clamp((diff ** 2).sum(dim=-1), min=0.0))

def pearson_corr_torch(x, y):
    x = x - x.mean()
    y = y - y.mean()
    return (x * y).mean() / (x.std() * y.std() + 1e-8)

def rank_torch(x):
    return torch.argsort(torch.argsort(x))

def spearman_corr_torch(x, y):
    rx = rank_torch(x).float()
    ry = rank_torch(y).float()
    return pearson_corr_torch(rx, ry)

def compute_distance_metrics_torch(X, Z, device="cpu"):
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    Z_t = torch.tensor(Z, dtype=torch.float32, device=device)
    D_X = pairwise_distances_torch(X_t)
    D_Z = pairwise_distances_torch(Z_t)
    n = D_X.shape[0]
    idx = torch.triu_indices(n, n, offset=1)
    d_x = D_X[idx[0], idx[1]]
    d_z = D_Z[idx[0], idx[1]]
    corr = pearson_corr_torch(d_x, d_z)
    rank_corr = spearman_corr_torch(d_x, d_z)
    return corr.item(), rank_corr.item()



# Optimizing Gini MDS

def optimize_gini(X, nu_grid, T=2):
    nu = 2.0
    final_Z = None
    final_time = None

    for _ in range(T):
        best_nu = None
        best_stress = np.inf
        
        for nu_candidate in nu_grid:
            gini = GiniMDS(
                n_components=2,
                nu=nu_candidate,
                mds_method='sammon',
                max_iter=50
            )
            start = time.perf_counter()
            Z = gini.fit_transform(X)
            elapsed = time.perf_counter() - start
            D = gini.gini_distances(X)
            stress = gini.stress(D, Z, kind='sammon')
            if stress < best_stress:
                best_stress = stress
                best_nu = nu_candidate
                final_Z = Z
                final_time = elapsed
        nu = best_nu
    return nu, final_Z, final_time



# Grid search

def tune_tsne(X, grid):
    best_score, best_Z = -np.inf, None
    best_time = None
    for p in grid:
        start = time.perf_counter()
        Z = TSNE(n_components=2, perplexity=p, init='pca').fit_transform(X)
        elapsed = time.perf_counter() - start
        score = np.var(Z)
        if score > best_score:
            best_score, best_Z = score, Z
            best_time = elapsed
    return best_Z, best_time

def tune_isomap(X, grid):
    best_score, best_Z = -np.inf, None
    best_time = None
    for k in grid:
        start = time.perf_counter()
        Z = Isomap(n_neighbors=k, n_components=2).fit_transform(X)
        elapsed = time.perf_counter() - start
        score = np.var(Z)
        if score > best_score:
            best_score, best_Z = score, Z
            best_time = elapsed
    return best_Z, best_time

def tune_umap(X, grid):
    best_score, best_Z = -np.inf, None
    best_time = None
    for k in grid:
        start = time.perf_counter()
        Z = umap.UMAP(n_neighbors=k, n_components=2).fit_transform(X)
        elapsed = time.perf_counter() - start
        score = np.var(Z)
        if score > best_score:
            best_score, best_Z = score, Z
            best_time = elapsed
    return best_Z, best_time



# Main XP

def run_experiment(n_runs=100):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    methods = ['tsne', 'isomap', 'umap', 'gini']
    scores = {
        m: {
            'trust': [], 'sil': [], 'nn': [],
            'corr': [], 'rank_corr': [], 'time': []
        }
        for m in methods
    }
    nu_values = []
    nu_grid = np.arange(1.5, 3.51, 0.1)

    for i in range(n_runs):
        print(f"Iteration {i+1}/{n_runs}")
        X, y = generate_heavy_tailed_data()

        # Gini
        best_nu, Z_gini, t_gini = optimize_gini(X, nu_grid)
        scores['gini']['time'].append(t_gini)
        nu_values.append(best_nu)

        # t-SNE (optimized)
        Z_tsne, t_tsne = tune_tsne(X, [10, 30, 50])
        scores['tsne']['time'].append(t_tsne)

        # Isomap (optimized)
        Z_iso, t_iso = tune_isomap(X, [5, 10, 15])
        scores['isomap']['time'].append(t_iso)

        # UMAP (optimized)
        Z_umap, t_umap = tune_umap(X, [10, 15, 30])
        scores['umap']['time'].append(t_umap)

        # Eval
        embeddings = {
            'tsne': Z_tsne,
            'isomap': Z_iso,
            'umap': Z_umap,
            'gini': Z_gini
        }
        for name, Z in embeddings.items():
            trust, sil, nn = compute_metrics(X, Z, y)
            corr, rank_corr = compute_distance_metrics_torch(X, Z, device)
            scores[name]['trust'].append(trust)
            scores[name]['sil'].append(sil)
            scores[name]['nn'].append(nn)
            scores[name]['corr'].append(corr)
            scores[name]['rank_corr'].append(rank_corr)

    # Print results
    results = {}
    for m in methods:
        results[m] = {
            'trust': np.mean(scores[m]['trust']),
            'sil': np.mean(scores[m]['sil']),
            'nn': np.mean(scores[m]['nn']),
            'corr': np.mean(scores[m]['corr']),
            'rank_corr': np.mean(scores[m]['rank_corr']),
            'time': np.mean(scores[m]['time']),
            'time_std': np.std(scores[m]['time'])
        }

    print("\n=== Hyper-parameter Nu (Gini) ===")
    print("Mean nu:", np.mean(nu_values))
    print("Std nu:", np.std(nu_values))

    df = pd.DataFrame(results).T  
    df = df.round(4)
    print("\n=== Results ===\n")
    print(df)
    return df



# Run
results = run_experiment(n_runs=500)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_heatmap(df):

    plt.figure(figsize=(8, 4))

    sns.heatmap(
        df,
        annot=True,
        cmap="coolwarm",
        fmt=".3f",
        linewidths=0.5
    )

    plt.title("Comparison of Dimensionality Reduction Methods")
    plt.tight_layout()
    plt.show()

df = results.copy()

df = df.rename(index={
    "tsne": "t-SNE",
    "isomap": "Isomap",
    "umap": "UMAP",
    "gini": "Gini MDS"
})

df = df.rename(columns={
    "trust": "Trust",
    "sil": "Silhouette",
    "nn": "NN",
    "corr": "Correl",
    "rank_corr": "Rank Correl",
    "time": "Time",
    "time_std": "Time Std"
})

plot_heatmap(df[['Trust','Silhouette','NN','Correl','Rank Correl']])

In [ ]:
print(df)

=== NU STATISTICS (GINI) ===
Mean nu: 1.5
Std nu: 0.0

=== RESULTS TABLE ===

         trust     sil      nn    corr  rank_corr    time  time_std
tsne    0.9781  0.0867  0.7648  0.3700     0.5621  3.3383    0.3729
isomap  0.8022  0.0181  0.5779  0.8274     0.7860  0.1468    0.0140
umap    0.9337  0.0995  0.7581  0.3278     0.5486  1.0911    0.3670
gini    0.7962  0.0474  0.6068  0.8625     0.8246  2.0045    0.2741
